# A05 · Vibe Coding the MADA Project

**IMB 2026 · MADA · Workshop A05 (Wed 1 Jul 2026, 9:00–12:00) — presentations are Friday.**

Monday you verified an AI's *words*. Today the AI writes **code** for you — and you
stay in charge. This is "vibe coding" done with discipline:

> **specify → prompt → run → review → iterate**

**Your AI surface today** *(free options only)*: Colab's built-in **Gemini chat panel**
(spark icon, bottom-left) — or the tool you used in Vibe coding 1 on Monday. If your
tool has a small daily request budget (e.g. Antigravity's free tier), treat prompts
like money: **plan before you spend.**

By the end your group will have: a working, **verified** slice of your MADA project,
plus an honest prompt log you can show on Friday.

In [ ]:
import os
import pandas as pd

DATA_BASE_URL = ""  # INSTRUCTOR: paste the published data folder URL here before class (see instructor-notes.md).

def load_file(name, reader=pd.read_csv, **kw):
    """Load a course data file: published URL -> local file -> Colab upload."""
    if DATA_BASE_URL:
        return reader(DATA_BASE_URL.rstrip("/") + "/" + name, **kw)
    if os.path.exists(name):
        return reader(name, **kw)
    try:
        from google.colab import files
        print(f"'{name}' not found - please upload it now (from this workshop's data/ folder).")
        files.upload()
        return reader(name, **kw)
    except ImportError as exc:
        raise FileNotFoundError(name) from exc

orders = load_file("orders.csv")
print("warm-up data:", orders.shape)

## 1 · The disciplined warm-up

Vibes produce code; **specs** produce *useful* code. A five-line spec template:

```
INPUT:   which file/data, which columns
TASK:    what transformation, step by step if it matters
OUTPUT:  exactly what should come out (table? chart? number?)
EXAMPLE: one concrete expected result, if you know one
CONSTRAINTS: libraries, style, what NOT to do
```

In [ ]:
# ✏️ TODO: write a 5-line spec (as a comment or string) for: "a monthly revenue summary report from orders.csv" — include the duplicate-rows issue you know about
# Hint: fill the five template lines; mention quantity * unit_price



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
spec = """
INPUT:   orders.csv - columns order_id, order_date, customer_id, product_id,
         product_name, category, quantity, unit_price. Contains some duplicate rows.
TASK:    drop duplicate rows; compute revenue = quantity * unit_price;
         aggregate revenue by calendar month of order_date.
OUTPUT:  a table month -> total revenue (chronological), plus a line chart,
         plus the grand total printed.
EXAMPLE: one row like  2024-12 | 96505.68
CONSTRAINTS: pandas + matplotlib only; chronological month order, not alphabetical.
"""
print(spec)
```

</details>

Now **prompt your AI tool with your spec** (paste the spec, ask for the code),
paste the generated code into the empty cell below, and run it.

In [ ]:
# ⬇️ paste the AI-generated code from YOUR spec here, then run


**Before you believe it — verify.** You know two numbers from A01: total FY
revenue ≈ **€815,630** and best month **December 2024** (≈ €96,506).

In [ ]:
# ✏️ TODO: check the AI code's output against the two anchor numbers (compute them independently here)
# Hint: drop_duplicates, revenue column, groupby month — you have done this before



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
# The two anchor numbers from A01 (after dropping duplicate rows):
#   total FY revenue ~= 815,629.71 EUR
#   best month       =  December 2024 (~96,505.68 EUR)
check = orders.drop_duplicates().copy()
check["revenue"] = check["quantity"] * check["unit_price"]
print("total revenue:", round(check["revenue"].sum(), 2))
monthly = check.groupby(pd.to_datetime(check["order_date"]).dt.to_period("M"))["revenue"].sum()
print("best month:   ", monthly.idxmax(), "->", round(monthly.max(), 2))
```

</details>

**The review checklist** — apply it to *every* AI-generated snippet, forever:

1. **Does it run?** (lowest bar, not the goal)
2. **Right numbers on a case you know?** (anchors!)
3. **Did it invent anything?** (columns, files, business rules)
4. **Anything unused or pointless?** (dead code = misunderstood task)

## 2 · Code-review drill

Below is a snippet "an AI generated" for exactly your warm-up task. It runs without
errors and produces a chart. **It is wrong twice.** Run it, apply the checklist,
find both flaws *before* opening the solution.

In [ ]:
# --- "AI-generated" monthly revenue summary --- (it RUNS without errors. Review it.)
import matplotlib.pyplot as plt

summary = orders.copy()
summary["month"] = pd.to_datetime(summary["order_date"]).dt.strftime("%B")
monthly_revenue = summary.groupby("month")["quantity"].sum()
monthly_revenue.plot(kind="bar", title="Adriatica monthly revenue (FY2024/25)")
plt.ylabel("revenue (EUR)")
plt.show()
print("total 'revenue':", monthly_revenue.sum())

In [ ]:
# ✏️ TODO: write down the two flaws (and the fix for each) as comments
# Hint: checklist item 2: compare with your anchor numbers; then look hard at the x-axis



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
# Flaw 1 - WRONG METRIC: it sums `quantity` (units sold) but labels it "revenue".
#          The total printed is ~16,000 - nowhere near the ~815,630 EUR you verified.
# Flaw 2 - SCRAMBLED ORDER: strftime("%B") makes month NAMES, so the x-axis sorts
#          alphabetically (April, August, December, ...) - the trend is unreadable.
# (Bonus)- it never dropped the 30 duplicate rows.
# The fix:
fixed = orders.drop_duplicates().copy()
fixed["revenue"] = fixed["quantity"] * fixed["unit_price"]
fixed["month"] = pd.to_datetime(fixed["order_date"]).dt.to_period("M")
monthly_fixed = fixed.groupby("month")["revenue"].sum()
print("total revenue (fixed):", round(monthly_fixed.sum(), 2))
```

</details>

**The lesson:** AI code fails *quietly*. It ran, it plotted, it printed — and both
headline numbers were garbage. *Running is not the same as right.* Your anchors caught
it in seconds.

## 3 · Recover, don't despair

Sometimes AI code crashes outright. That is the *easy* case — if you feed the error
back **with context**. The cell below is set NOT to run by default; flip the flag,
experience the crash, flip it back.

In [ ]:
RUN_BROKEN = False  # <- flip to True, run, and experience the crash (then flip back)

if RUN_BROKEN:
    # --- another "AI-generated" snippet, fresh from a chatbot ---
    df = load_file("orders.csv")
    df["profit"] = df["revenue"] - df["cost"]      # invented columns!
    print(df.groupby("region")["profit"].sum())   # invented column again
else:
    print("(set RUN_BROKEN = True to run the broken snippet)")

In [ ]:
# ✏️ TODO: write the feedback prompt you would send the AI: include the error, the real columns, and what you actually want
# Hint: error + context + expectation



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
# A good feedback prompt contains: the ERROR, the CONTEXT, the EXPECTATION.
feedback_prompt = """Your code raised:
    KeyError: 'revenue'
The DataFrame comes from orders.csv with columns:
    order_id, order_date, customer_id, product_id, product_name, category, quantity, unit_price
There is no revenue, cost, or region column. Revenue must be computed as
quantity * unit_price; there is no cost data, so drop the profit idea and
summarise revenue by product category instead."""
print(feedback_prompt)
```

</details>

**Recovery loop:** reproduce → feed back (error + context + expectation) → re-run
→ re-verify. **Restart rule:** if two recovery rounds haven't fixed it, your *spec*
was the problem — rewrite the spec, start a fresh chat.

## 4 · Your project slice — main block (~80 minutes)

Adriatica's chapter is closed. Now: **your MADA project**, Friday's demo, one thin
slice, built today, with AI doing the typing and your group doing the thinking.

**Scope it small.** One input you already have, one output, one screenshot-able
result. "The data part of our demo" — not "our whole project".

**📋 Spec card** *(double-click, fill in as a group)*

> **Project:** …
> **The slice we build today:** …
> **Input data:** … (file, columns)
> **Output:** … (table / chart / number)
> **Demo line for Friday:** "And here you can see …"


**🪵 Prompt log** — keep it honest, fill it as you go *(double-click to edit)*:

| # | What we asked the AI | What came back | How we verified it |
|---|---------------------|----------------|--------------------|
| 1 | … | … | … |
| 2 | … | … | … |

*(This log is your AI-use declaration for the project — same norm as the homework.
On Friday it shows you used AI like professionals: specified, reviewed, verified.)*

**Work plan for the block:**

1. Spec card (10 min — the thinking happens here)
2. Generate → run → **verify against a number you can hand-check** (the A01/A04 habit)
3. Iterate; log every prompt
4. If the tool budget is tight: draft prompts in the doc first, spend them deliberately

Instructors are circulating — call us at the *spec* stage, not only when code breaks.

## 5 · Dry run & wrap-up

**Dry run (last 20 min):** run your slice top-to-bottom as it will run on Friday.
Then **swap notebooks with a neighbouring group**: they check ONE of your numbers
by hand (and you theirs). If it survives a stranger's check, it will survive the demo.

---

### The course, in one breath

**A01** you made data answer questions · **A02** you made it predict · **A03** you
learned when models lie · **A04** you learned when *AI* lies · **A05** the AI worked
for you — and you verified everything. That habit — *trust but verify* — is the whole
strand in three words, and it is what managing AI-assisted teams will demand of you.

**Beyond this course:** deploy your slice as a small app; forecast the monthly trend
(time series); script LLM workflows over real document volumes. The toolkit you have
now is enough to start any of them. Good luck on Friday! 🎓